# Pobieranie i scalanie danych pomiarowych ze stacji naziemnych IMGW (Balice, Obserwatorium)

## Konfiguracja i importowanie bibliotek

In [ ]:
import os
import sys
import zipfile
from pathlib import Path
import pandas as pd
import requests

# Dodanie katalogu głównego projektu do sys.path (notatnik jest w notebooks/)
root_dir = Path("..").resolve()
if str(root_dir) not in sys.path:
  sys.path.append(str(root_dir))

# Import ścieżki surowych danych ze wspólnego modułu konfiguracyjnego
from src.config import IMGW_DATA_DIR, BALICE_PATH, OBSERWATORIUM_PATH

# Przygotowanie katalogów roboczych na dane IMGW
BALICE_DIR = IMGW_DATA_DIR / "balice"
OBSERWATORIUM_DIR = IMGW_DATA_DIR / "obserwatorium"

BALICE_DIR.mkdir(parents=True, exist_ok=True)
OBSERWATORIUM_DIR.mkdir(parents=True, exist_ok=True)
print("Katalogi dla danych IMGW zostały przygotowane.")

## Pobieranie i scalanie danych


### Kraków - Balice (stacja synoptyczna)

In [ ]:
station_id = "566"
station_name = "KRAKOW_BALICE"
base_url = "https://danepubliczne.imgw.pl/data/dane_pomiarowo_obserwacyjne/dane_meteorologiczne/terminowe/synop/"
start_year = 2000
end_year = 2025

print("--- 1. POBIERANIE DANYCH DLA STACJI KRAKÓW-BALICE ---")
for year in range(start_year, end_year + 1):
    year_file = BALICE_DIR / f"{station_name}_{year}.csv"
    if year_file.exists():
        print(f"Pominięto: {year_file.name}. Plik już istnieje.")
        continue

    year_folder = "1996_2000" if 1996 <= year <= 2000 else str(year)
    zip_filename = f"1996_2000_{station_id}_s.zip" if year == 2000 else f"{year}_{station_id}_s.zip"
    url = f"{base_url}{year_folder}/{zip_filename}"

    try:
        r = requests.get(url, timeout=25)
        if r.status_code == 200:
            temp_zip = BALICE_DIR / f"temp_{zip_filename}"
            with open(temp_zip, "wb") as f:
                f.write(r.content)

            with zipfile.ZipFile(temp_zip, "r") as z:
                csvs = [f for f in z.namelist() if station_id in f and f.endswith(".csv")]
                if csvs:
                with z.open(csvs[0]) as f_csv:
                    df = pd.read_csv(f_csv, header=None, encoding="iso-8859-2", low_memory=False)
                    # Kolumny IMGW Synop: 2=rok, 3=miesiąc, 4=dzień, 5=godzina, 29=temperatura
                    df_sub = df[[2, 3, 4, 5, 29]].copy()
                    df_sub.columns = ["year", "month", "day", "hour", "temp"]
                    df_sub["temp"] = pd.to_numeric(df_sub["temp"], errors="coerce")
                    df_year = df_sub[df_sub["year"] == year].copy()

                    df_year.to_csv(year_file, index=False)
                    print(f"Balice {year}: zapisano {year_file.name} ({len(df_year)} wierszy)")
            temp_zip.unlink(missing_ok=True)
        else:
            print(f"Balice {year}: brak na serwerze (kod {r.status_code})")
    except Exception as e:
        print(f"Balice {year}: błąd ({e})")

print("Zakończono pobieranie danych dla Balic.")

# Scalanie plików rocznych Balic do jednego CSV
yearly_files = sorted(BALICE_DIR.glob(f"{station_name}_*.csv"))
if yearly_files:
    df_all = pd.concat([pd.read_csv(f) for f in yearly_files], ignore_index=True)
    df_all.sort_values(by=['year', 'month', 'day', 'hour'], inplace=True)
    df_all.to_csv(BALICE_PATH, index=False)
    print(f"Utworzono plik zbiorczy Balic: {BALICE_PATH} ({len(df_all):,} wierszy)\n")

### Scalanie plików 